# TAM Research compute connection

Clone the public repo, install its Python package, and verify this Colab runtime.
Start with CPU. No training, experiment queue, GitHub credential, or Drive mount
is used. Pair Google's official Colab MCP bridge with the local desktop agent
to let the agent edit cells, execute code, and read results here.

See [the connection guide](https://github.com/vinceackermann2-sys/tam-research/blob/main/docs/COLAB_COMPUTE.md).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/vinceackermann2-sys/tam-research.git"
SOURCE_REF = "main"  # A branch, tag, or exact Git commit; change explicitly for a PR.
REPO_DIR = Path("/content/tam-research")
RESULTS_DIR = Path("/content/tam-results")

def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

if REPO_DIR.exists():
    if git("remote", "get-url", "origin") != REPOSITORY:
        raise RuntimeError("Existing checkout has another origin; choose a new REPO_DIR.")
    if git("status", "--porcelain"):
        raise RuntimeError("Save local edits before changing this checkout.")
else:
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", SOURCE_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = git("rev-parse", "HEAD")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
print("Repository:", REPOSITORY)
print("Pinned commit:", COMMIT)
print("Results directory:", RESULTS_DIR)


In [ ]:
# Install core repo dependencies. No experiment is launched.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
print("TAM Research package installed")


In [ ]:
# A tiny compute check; GPU is used only if one is already allocated.
subprocess.run([
    sys.executable, str(REPO_DIR / "scripts" / "colab_compute_smoke.py"),
    "--output", str(RESULTS_DIR / "connection-smoke.json"),
], check=True, cwd=REPO_DIR)


## Run a bounded repo command

The desktop agent can add and execute cells here after pairing. Choose the
experiment and data explicitly before a research run. CUDA training uses a GPU
runtime; TPU requires code adaptation. Runtime resets require rerunning setup.
Modal data and checkpoints are not copied by this notebook.

Save outputs under `/content/tam-results`. Download them before the runtime is
deleted; files in `/content` are temporary. The optional next cell downloads
the results folder as a ZIP.


In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive("/content/tam-results-export", "zip", RESULTS_DIR)
files.download(archive)
